In [ ]:
!pip install pymc
!pip install arviz


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

import scipy.stats as stats
from scipy.optimize import minimize


from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("example.csv")
x_axis = np.arange(df.shape[0])

# Interpolate using cubic spline, then clip negative values to 0
df_clean = df.iloc[:, 1:].interpolate(method='cubic', limit=3).clip(lower=0)

sus = (df_clean.isna().sum(axis=0)).sort_values()[-4:].index
df_clean.drop(columns=sus, inplace=True)

df_clean = df_clean.fillna(df_clean.mean(numeric_only=True))
df_clean['target'] = df_clean.sum(axis=1) + np.random.normal(
    loc=0,
    scale=5,
    size=df_clean.sum(axis=1).shape
)

In [ ]:
def distort_columns_by_multiplier(X, col_share=0.2, random_state=None):
    """Домножает случайные столбцы матрицы X на случайный коэффициент из [0, 1)."""
    if random_state is not None:
        np.random.seed(random_state)

    X_distorted = X.copy()
    n_samples, n_features = X_distorted.shape

    # Находим индексы столбцов, которые будем искажать
    n_distort = int(n_features * col_share)
    distort_indices = np.random.choice(n_features, size=n_distort, replace=False)

    # Генерируем случайные коэффициенты для каждого выбранного столбца
    multipliers = np.random.uniform(0.0, 1.0, size=n_distort)

    # Применяем умножение
    if isinstance(X_distorted, pd.DataFrame):
        for idx, mult in zip(distort_indices, multipliers):
            X_distorted.iloc[:, idx] = X_distorted.iloc[:, idx] * mult
    else:
        for idx, mult in zip(distort_indices, multipliers):
            X_distorted[:, idx] = X_distorted[:, idx] * mult

    return X_distorted

In [ ]:
def zero_consecutive_elements(X, col_share=0.2, drop_share=0.1, random_state=None):
    """В случайных столбцах зануляет заданную долю элементов, идущих подряд."""
    if random_state is not None:
        np.random.seed(random_state)

    X_distorted = X.copy()
    n_samples, n_features = X_distorted.shape

    # Сколько столбцов нужно исказить
    n_distort = int(n_features * col_share)
    distort_indices = np.random.choice(n_features, size=n_distort, replace=False)

    # Сколько элементов подряд нужно занулить в столбце
    block_size = max(1, int(n_samples * drop_share))

    # Преобразуем в numpy для быстрого индексирования, если это DataFrame
    is_df = isinstance(X_distorted, pd.DataFrame)
    arr = X_distorted.values.copy() if is_df else X_distorted.copy()

    for col_idx in distort_indices:
        # Случайно выбираем начало блока по строкам
        max_start = n_samples - block_size
        start_row = (
            np.random.randint(0, max_start + 1) if max_start > 0 else 0
        )

        arr[start_row : start_row + block_size, col_idx] = 0.0

    return pd.DataFrame(arr, columns=X.columns, index=X.index) if is_df else arr

In [ ]:
def zero_every_kth_element(X, col_share=0.2, k=None, random_state=None):
    """В случайных столбцах обнуляет каждый k-й элемент.

    Если k=None, шаг выбирается случайно для каждого столбца.
    """
    if random_state is not None:
        np.random.seed(random_state)

    X_distorted = X.copy()
    n_samples, n_features = X_distorted.shape

    # Выбираем столбцы, которые будем искажать
    n_distort = int(n_features * col_share)
    distort_indices = np.random.choice(n_features, size=n_distort, replace=False)

    is_df = isinstance(X_distorted, pd.DataFrame)
    arr = X_distorted.values.copy() if is_df else X_distorted.copy()

    for idx in distort_indices:
        # Если k не задан, генерируем случайный шаг
        current_k = (
            k
            if k is not None
            else np.random.randint(2, max(3, n_samples // 2))
        )

        # Обнуляем элементы через заданный шаг по строкам
        arr[::current_k, idx] = 0.0

    return (
        pd.DataFrame(arr, columns=X.columns, index=X.index)
        if is_df
        else arr
    )

In [ ]:
x = df_clean.drop(columns='target')
y = df_clean['target']

# Model comparison and uncertainty evaluation

This section provides a common evaluation interface for constrained SciPy optimization, Linear Regression, Ridge, and the existing Bayesian PyMC model.

**Important:** all methods estimate correction weights `beta` in `y ~= X_dirty @ beta + intercept`. Convert a positive weight to an implied persistent scale with `alpha_hat = 1 / beta`. This conversion is only meaningful when the model assumptions match the corruption process. Negative or zero regression weights are reported as invalid alpha estimates rather than silently clipped.

For uncertainty, non-Bayesian methods use a moving-block bootstrap over time rows. This is preferable to an IID row bootstrap when neighboring meter readings are temporally correlated. Bootstrap intervals measure sampling variability under the fitted procedure; they do not automatically include model misspecification.

The runner stores both per-user estimates and one-row-per-method aggregate metrics so the results can be concatenated across synthetic trials.


In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

def fit_meter_alphas(
    X_dirty,
    y,
    *,
    valid_rows=None,
    noise_scale=5.0,
    prior_correction_scale=0.5,
    draws=2000,
    tune=2000,
    chains=4,
    target_accept=0.95,
    random_seed=42,
):
    if isinstance(X_dirty, pd.DataFrame):
        feature_names = list(X_dirty.columns)
        X_arr = X_dirty.to_numpy(dtype=float)
    else:
        X_arr = np.asarray(X_dirty, dtype=float)
        feature_names = [f"user_{j}" for j in range(X_arr.shape[1])]

    y_arr = np.asarray(y, dtype=float).reshape(-1)

    if X_arr.ndim != 2:
        raise ValueError("X_dirty must be a 2D array or DataFrame.")
    if y_arr.shape[0] != X_arr.shape[0]:
        raise ValueError("X_dirty and y must have the same number of rows.")
    if not np.isfinite(X_arr).all() or not np.isfinite(y_arr).all():
        raise ValueError("X_dirty and y must contain only finite values.")
    if (X_arr < 0).any():
        raise ValueError("Meter readings must be non-negative.")
    if noise_scale <= 0 or prior_correction_scale <= 0:
        raise ValueError("noise_scale and prior_correction_scale must be positive.")

    n_times, n_users = X_arr.shape
    if valid_rows is None:
        mask = np.ones(n_times, dtype=bool)
    else:
        mask = np.asarray(valid_rows, dtype=bool)
        if mask.shape != (n_times,):
            raise ValueError("valid_rows must be a boolean vector with one value per row.")

    X_fit = X_arr[mask]
    y_fit = y_arr[mask]
    if X_fit.shape[0] < max(10, n_users // 2):
        raise ValueError(
            f"Only {X_fit.shape[0]} rows remain for {n_users} users. "
            "Use more valid rows or reduce the model complexity."
        )

    # Columns that are zero throughout the fitting data cannot identify beta.
    all_zero_in_fit = np.all(X_fit == 0, axis=0)
    if all_zero_in_fit.all():
        raise ValueError("All user columns are zero in the selected fitting rows.")

    with pm.Model() as model:
        # beta=1 means no underestimation; beta>1 compensates for low readings.
        correction = pm.HalfNormal(
            "correction",
            sigma=prior_correction_scale,
            shape=n_users,
        )
        beta = pm.Deterministic("beta", 1.0 + correction)

        sigma = pm.HalfNormal("sigma", sigma=noise_scale)
        mu = pm.math.sum(X_fit * beta[None, :], axis=1)

        pm.Normal("central_reading", mu=mu, sigma=sigma, observed=y_fit)

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            return_inferencedata=True,
        )

    beta_samples = (
        idata.posterior["beta"]
        .stack(sample=("chain", "draw"))
        .transpose("sample", "beta_dim_0")
        .values
    )
    alpha_samples = 1.0 / beta_samples

    rows = []
    for j, name in enumerate(feature_names):
        if all_zero_in_fit[j]:
            rows.append({
                "user_id": name,
                "alpha_median": np.nan,
                "alpha_lower_95": np.nan,
                "alpha_upper_95": np.nan,
                "p_underestimated_5pct": np.nan,
                "p_underestimated_20pct": np.nan,
                "status": "unidentifiable: all zero in fitting rows",
            })
            continue

        samples = alpha_samples[:, j]
        lo, median, hi = np.quantile(samples, [0.025, 0.5, 0.975])
        rows.append({
            "user_id": name,
            "alpha_median": median,
            "alpha_lower_95": lo,
            "alpha_upper_95": hi,
            "p_underestimated_5pct": np.mean(samples < 0.95),
            "p_underestimated_20pct": np.mean(samples < 0.80),
            "status": "estimate; check interval width and diagnostics",
        })

    result = pd.DataFrame(rows)
    diagnostics = az.summary(idata, var_names=["beta", "sigma"], hdi_prob=0.95)
    return result, idata, diagnostics


In [ ]:

import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score


def fit_scipy_bounded(X, y, *, beta_min=1.0, beta_max=15.0,
                      fit_intercept=False, intercept_bounds=(-10.0, 10.0),
                      maxiter=2000):
    """Least-squares fit with box constraints on beta and optional intercept."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    n_features = X.shape[1]

    if fit_intercept:
        initial = np.r_[np.clip(np.ones(n_features), beta_min, beta_max), 0.0]
        bounds = [(beta_min, beta_max)] * n_features + [intercept_bounds]
        def predict(params):
            return X @ params[:-1] + params[-1]
    else:
        initial = np.clip(np.ones(n_features), beta_min, beta_max)
        bounds = [(beta_min, beta_max)] * n_features
        def predict(params):
            return X @ params

    def objective(params):
        residual = y - predict(params)
        return np.mean(residual ** 2)

    result = minimize(
        objective, initial, method="L-BFGS-B", bounds=bounds,
        options={"maxiter": maxiter}
    )
    params = result.x
    beta = params[:-1] if fit_intercept else params
    intercept = float(params[-1]) if fit_intercept else 0.0
    return {
        "beta": np.asarray(beta),
        "intercept": intercept,
        "y_pred": predict(params),
        "success": bool(result.success),
        "message": str(result.message),
        "objective": float(result.fun),
    }


def fit_sklearn_linear(X, y, *, fit_intercept=False, positive=False):
    model = LinearRegression(fit_intercept=fit_intercept, positive=positive)
    model.fit(X, y)
    beta = np.asarray(model.coef_, dtype=float).reshape(-1)
    intercept = float(model.intercept_) if fit_intercept else 0.0
    return {
        "beta": beta, "intercept": intercept,
        "y_pred": model.predict(X), "success": True, "message": "ok"
    }


def fit_sklearn_ridge(X, y, *, ridge_penalty=1.0,
                      fit_intercept=False, positive=False):
    model = Ridge(alpha=ridge_penalty, fit_intercept=fit_intercept,
                  positive=positive)
    model.fit(X, y)
    beta = np.asarray(model.coef_, dtype=float).reshape(-1)
    intercept = float(model.intercept_) if fit_intercept else 0.0
    return {
        "beta": beta, "intercept": intercept,
        "y_pred": model.predict(X), "success": True, "message": "ok"
    }



def fit_pymc_bayesian(X, y, *, draws=500, tune=500, chains=2,
                      random_seed=42, noise_scale=5.0,
                      prior_correction_scale=0.5):
    """
    Bayesian estimator using the fit_meter_alphas function defined earlier.
    Returns posterior mean beta and posterior alpha interval as extra fields.
    """
    alpha_table, idata, diagnostics = fit_meter_alphas(
        X, y,
        noise_scale=noise_scale,
        prior_correction_scale=prior_correction_scale,
        draws=draws, tune=tune, chains=chains,
        random_seed=random_seed,
    )
    beta_samples = (
        idata.posterior["beta"]
        .stack(sample=("chain", "draw"))
        .transpose("sample", "beta_dim_0")
        .values
    )
    beta_mean = beta_samples.mean(axis=0)
    return {
        "beta": beta_mean,
        "intercept": 0.0,
        "y_pred": np.asarray(X, dtype=float) @ beta_mean,
        "success": True,
        "message": "PyMC posterior mean beta",
        "alpha_posterior_lower_95": alpha_table["alpha_lower_95"].to_numpy(),
        "alpha_posterior_upper_95": alpha_table["alpha_upper_95"].to_numpy(),
        "alpha_posterior_median": alpha_table["alpha_median"].to_numpy(),
        "posterior_diagnostics": diagnostics,
    }


def default_comparison_methods(beta_min=1.0, beta_max=15.0, include_bayesian=False):
    """Consistent method registry. Each callable returns beta/intercept/prediction."""
    methods = {
        "OLS_no_intercept": lambda X, y: fit_sklearn_linear(
            X, y, fit_intercept=False, positive=False),
        "OLS_positive_no_intercept": lambda X, y: fit_sklearn_linear(
            X, y, fit_intercept=False, positive=True),
        "OLS_positive_intercept": lambda X, y: fit_sklearn_linear(
            X, y, fit_intercept=True, positive=True),
        "Ridge_0.1_no_intercept": lambda X, y: fit_sklearn_ridge(
            X, y, ridge_penalty=0.1, fit_intercept=False, positive=False),
        "Ridge_1_no_intercept": lambda X, y: fit_sklearn_ridge(
            X, y, ridge_penalty=1.0, fit_intercept=False, positive=False),
        "Ridge_10_no_intercept": lambda X, y: fit_sklearn_ridge(
            X, y, ridge_penalty=10.0, fit_intercept=False, positive=False),
        "Ridge_positive_1_no_intercept": lambda X, y: fit_sklearn_ridge(
            X, y, ridge_penalty=1.0, fit_intercept=False, positive=True),
        "L-BFGS-B_beta_bounds_no_intercept": lambda X, y: fit_scipy_bounded(
            X, y, beta_min=beta_min, beta_max=beta_max,
            fit_intercept=False),
    }
    if include_bayesian:
        methods["PyMC_Bayesian"] = lambda X, y: fit_pymc_bayesian(
            X, y, draws=500, tune=500, chains=2, random_seed=42)
    return methods


def _moving_block_indices(n, block_length, rng):
    """Sample n indices by concatenating randomly selected contiguous blocks."""
    block_length = max(1, min(int(block_length), n))
    starts = rng.integers(0, max(1, n - block_length + 1),
                          size=int(np.ceil(n / block_length)))
    idx = np.concatenate([
        np.arange(s, min(s + block_length, n)) for s in starts
    ])
    return idx[:n]


def compare_alpha_estimators(
    X_dirty, y, alpha_true, *,
    methods=None,
    bootstrap_reps=100,
    block_length=24,
    random_state=42,
    evaluation_mask=None,
):
    """
    Fit estimators and assess implied alpha against known synthetic truth.

    `evaluation_mask` selects columns whose alpha values are identifiable and
    should contribute to alpha-recovery metrics (e.g. exclude truly failed
    meters with alpha_true == 0). All fitted columns are still used by the model.

    Returns
    -------
    method_metrics : one row per method
    user_results : one row per method and user
    Notes:
      * `alpha_true` should be the persistent scaling factor used by the
        corruption generator, not calculated as clean/dirty for zeroed rows.
      * alpha_hat=1/beta_hat is reported only for finite beta_hat > 0.
      * Bootstrap CI is percentile interval from moving-block bootstrap.
      * Predictive metrics here are in-sample, so use a separate time holdout
        if out-of-sample prediction is an evaluation goal.
    """
    is_frame = isinstance(X_dirty, pd.DataFrame)
    names = list(X_dirty.columns) if is_frame else [
        f"user_{j}" for j in range(np.asarray(X_dirty).shape[1])]
    X = np.asarray(X_dirty, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    truth = np.asarray(alpha_true, dtype=float).reshape(-1)

    if X.ndim != 2 or len(y) != X.shape[0]:
        raise ValueError("X_dirty must be 2D and have the same row count as y.")
    if len(truth) != X.shape[1]:
        raise ValueError("alpha_true must have one value per user/column.")
    if evaluation_mask is None:
        eval_mask = np.ones(X.shape[1], dtype=bool)
    else:
        eval_mask = np.asarray(evaluation_mask, dtype=bool).reshape(-1)
        if len(eval_mask) != X.shape[1]:
            raise ValueError("evaluation_mask must have one value per user/column.")
    if methods is None:
        methods = default_comparison_methods()

    rng = np.random.default_rng(random_state)
    metrics_rows, user_rows = [], []

    for method_name, fit_func in methods.items():
        start = time.perf_counter()
        try:
            fit = fit_func(X, y)
            beta = np.asarray(fit["beta"], dtype=float).reshape(-1)
            intercept = float(fit["intercept"])
            pred = np.asarray(fit.get("y_pred", X @ beta + intercept))
            alpha_hat = np.divide(1.0, beta, out=np.full_like(beta, np.nan),
                                   where=np.isfinite(beta) & (beta > 0))

            boot_alpha = []
            boot_beta = []
            boot_failures = 0
            # Bayesian fits already provide posterior intervals; do not run an
            # expensive outer bootstrap around MCMC by default.
            if "alpha_posterior_lower_95" not in fit:
                for _ in range(int(bootstrap_reps)):
                    idx = _moving_block_indices(len(y), block_length, rng)
                    try:
                        bfit = fit_func(X[idx], y[idx])
                        b = np.asarray(bfit["beta"], dtype=float).reshape(-1)
                        a = np.divide(1.0, b, out=np.full_like(b, np.nan),
                                      where=np.isfinite(b) & (b > 0))
                        boot_beta.append(b)
                        boot_alpha.append(a)
                    except Exception:
                        boot_failures += 1

            if boot_alpha:
                boot_alpha = np.asarray(boot_alpha)
                alpha_lo = np.nanquantile(boot_alpha, 0.025, axis=0)
                alpha_hi = np.nanquantile(boot_alpha, 0.975, axis=0)
                beta_boot = np.asarray(boot_beta)
                beta_lo = np.nanquantile(beta_boot, 0.025, axis=0)
                beta_hi = np.nanquantile(beta_boot, 0.975, axis=0)
            else:
                alpha_lo = alpha_hi = beta_lo = beta_hi = np.full(X.shape[1], np.nan)

            # Bayesian intervals are posterior credible intervals, not bootstrap intervals.
            if "alpha_posterior_lower_95" in fit:
                alpha_lo = np.asarray(fit["alpha_posterior_lower_95"], dtype=float)
                alpha_hi = np.asarray(fit["alpha_posterior_upper_95"], dtype=float)
                alpha_hat = np.asarray(fit["alpha_posterior_median"], dtype=float)

            for j, user in enumerate(names):
                user_rows.append({
                    "method": method_name,
                    "user_id": user,
                    "beta_hat": beta[j],
                    "beta_boot_lower_95": beta_lo[j],
                    "beta_boot_upper_95": beta_hi[j],
                    "alpha_true": truth[j],
                    "alpha_hat": alpha_hat[j],
                    "alpha_boot_lower_95": alpha_lo[j],
                    "alpha_boot_upper_95": alpha_hi[j],
                    "alpha_evaluated": bool(eval_mask[j]),
                    "alpha_error": alpha_hat[j] - truth[j]
                        if eval_mask[j] else np.nan,
                    "alpha_abs_error": abs(alpha_hat[j] - truth[j])
                        if eval_mask[j] and np.isfinite(alpha_hat[j]) else np.nan,
                    "alpha_interval_covers_truth": bool(
                        eval_mask[j] and np.isfinite(alpha_lo[j])
                        and np.isfinite(alpha_hi[j])
                        and alpha_lo[j] <= truth[j] <= alpha_hi[j]),
                    "intercept_hat": intercept,
                    "fit_success": fit.get("success", True),
                })

            residual = y - pred
            metrics_rows.append({
                "method": method_name,
                "mse_in_sample": mean_squared_error(y, pred),
                "rmse_in_sample": mean_squared_error(y, pred) ** 0.5,
                "r2_in_sample": r2_score(y, pred),
                "alpha_mae": np.nanmean(np.abs(alpha_hat[eval_mask] - truth[eval_mask])),
                "alpha_rmse": np.sqrt(np.nanmean((alpha_hat[eval_mask] - truth[eval_mask]) ** 2)),
                "alpha_bias": np.nanmean(alpha_hat[eval_mask] - truth[eval_mask]),
                "alpha_interval_coverage": np.nanmean(
                    (alpha_lo[eval_mask] <= truth[eval_mask])
                    & (truth[eval_mask] <= alpha_hi[eval_mask])),
                "mean_alpha_interval_width": np.nanmean(
                    alpha_hi[eval_mask] - alpha_lo[eval_mask]),
                "bootstrap_reps_requested": int(bootstrap_reps),
                "bootstrap_failures": boot_failures,
                "runtime_seconds": time.perf_counter() - start,
                "fit_success": fit.get("success", True),
                "fit_message": fit.get("message", "ok"),
            })
        except Exception as exc:
            metrics_rows.append({
                "method": method_name,
                "mse_in_sample": np.nan, "rmse_in_sample": np.nan,
                "r2_in_sample": np.nan, "alpha_mae": np.nan,
                "alpha_rmse": np.nan, "alpha_bias": np.nan,
                "alpha_interval_coverage": np.nan,
                "mean_alpha_interval_width": np.nan,
                "bootstrap_reps_requested": int(bootstrap_reps),
                "bootstrap_failures": int(bootstrap_reps),
                "runtime_seconds": time.perf_counter() - start,
                "fit_success": False, "fit_message": repr(exc),
            })

    return pd.DataFrame(metrics_rows), pd.DataFrame(user_rows)


## Generate synthetic corruption with explicit ground truth

The generator can now create **persistently unworking meters** by setting entire columns to zero (`unworking_share`). Their true alpha is stored as `0.0`. Other corruptions (persistent scaling, temporary blocks, and periodic zeros) are applied to working meters. The central target `y` remains the total from the clean readings plus whatever noise was already included in `y`.

In [ ]:
def make_corrupted_meter_data(
    X_clean, *,
    unworking_share=0.05,
    scale_share=0.3,
    scale_range=(0.2, 0.9),
    block_share=0.05,
    block_fraction=0.20,
    periodic_share=0.05,
    every_k=3,
    random_state=42,
):
    """Return corrupted readings, true persistent alphas, and metadata.

    `unworking_share` selects permanently failed meters: their entire observed
    columns are set to zero and alpha_true is 0. The other corruption stages
    are applied only to the remaining working meters. Stage selections may
    overlap with one another.
    """
    is_df = isinstance(X_clean, pd.DataFrame)
    columns = list(X_clean.columns) if is_df else [
        f"user_{j}" for j in range(np.asarray(X_clean).shape[1])]
    X = np.asarray(X_clean, dtype=float).copy()
    n_rows, n_cols = X.shape
    if n_cols == 0:
        raise ValueError("X_clean must contain at least one meter column.")
    rng = np.random.default_rng(random_state)

    def sample_columns(pool, share):
        pool = np.asarray(pool, dtype=int)
        if len(pool) == 0 or share <= 0:
            return np.array([], dtype=int)
        count = min(len(pool), max(1, int(round(n_cols * share))))
        return rng.choice(pool, size=count, replace=False)

    all_cols = np.arange(n_cols)
    unworking_cols = sample_columns(all_cols, unworking_share)
    alpha_true = np.ones(n_cols, dtype=float)
    alpha_true[unworking_cols] = 0.0
    X[:, unworking_cols] = 0.0
    working_cols = np.setdiff1d(all_cols, unworking_cols)

    scaled_cols = sample_columns(working_cols, scale_share)
    multipliers = rng.uniform(scale_range[0], scale_range[1], size=len(scaled_cols))
    alpha_true[scaled_cols] = multipliers
    X *= alpha_true[None, :]

    block_mask = np.zeros_like(X, dtype=bool)
    block_cols = sample_columns(working_cols, block_share)
    block_size = max(1, int(n_rows * block_fraction)) if n_rows else 0
    for j in block_cols:
        if block_size > 0:
            start = int(rng.integers(0, max(1, n_rows - block_size + 1)))
            block_mask[start:start + block_size, j] = True
    X[block_mask] = 0.0

    periodic_mask = np.zeros_like(X, dtype=bool)
    periodic_cols = sample_columns(working_cols, periodic_share)
    if every_k is not None and every_k >= 1:
        for j in periodic_cols:
            periodic_mask[::every_k, j] = True
        X[periodic_mask] = 0.0

    dirty = pd.DataFrame(X, columns=columns, index=X_clean.index) if is_df else X
    metadata = {
        "unworking_columns": [columns[j] for j in unworking_cols],
        "unworking_mask": np.isin(all_cols, unworking_cols),
        "scaled_columns": [columns[j] for j in scaled_cols],
        "scale_multipliers": {columns[j]: alpha_true[j] for j in scaled_cols},
        "block_columns": [columns[j] for j in block_cols],
        "periodic_columns": [columns[j] for j in periodic_cols],
        "block_mask": block_mask,
        "periodic_mask": periodic_mask,
    }
    return dirty, alpha_true, metadata


def detect_unworking_meters(X_dirty, *, near_zero_fraction=0.98, abs_tol=1e-8):
    """Detect columns with at least `near_zero_fraction` near-zero readings.

    This intentionally targets persistent failures, not temporary blocks or
    periodic zeros. `abs_tol` should be chosen in the units of the meter data.
    Returns one row per meter with the detection diagnostics.
    """
    is_df = isinstance(X_dirty, pd.DataFrame)
    X = np.asarray(X_dirty, dtype=float)
    names = list(X_dirty.columns) if is_df else [f"user_{j}" for j in range(X.shape[1])]
    if X.ndim != 2:
        raise ValueError("X_dirty must be a 2D array or DataFrame.")
    if not 0 <= near_zero_fraction <= 1:
        raise ValueError("near_zero_fraction must be between 0 and 1.")
    near_zero = np.abs(X) <= abs_tol
    frac = near_zero.mean(axis=0) if X.shape[0] else np.ones(X.shape[1])
    max_abs = np.max(np.abs(X), axis=0) if X.shape[0] else np.zeros(X.shape[1])
    detected = frac >= near_zero_fraction
    return pd.DataFrame({
        "user_id": names,
        "near_zero_fraction": frac,
        "max_abs_reading": max_abs,
        "detected_unworking": detected,
    })


## Unified `run_auto_test`: original pipeline vs equal-share adjustment

Each trial runs two variants using the same corrupted data and detected failed meters:

1. **Unadjusted:** fit methods on all observed columns against the original central target.
2. **Equal-share baseline:** detect persistently near-zero columns, remove them from the predictors, and adjust the target using `y_adjusted = y * (1 - k_detected / n_total)`. Fit the same methods on the remaining columns.

Alpha-recovery metrics are evaluated only for truly working meters (`alpha_true > 0`), because a zeroed meter has no identifiable inverse multiplier. Detection precision/recall are reported separately. This is a synthetic benchmark: true labels are used only for evaluation, never for detection or adjustment.

In [ ]:
def run_auto_test(
    X_clean, y, *,
    methods=None,
    trial_id=0,
    corruption_kwargs=None,
    bootstrap_reps=100,
    block_length=24,
    random_state=42,
    near_zero_fraction=0.98,
    near_zero_abs_tol=1e-8,
):
    """Run unadjusted and equal-share failed-meter baselines on one trial.

    Equal-share baseline:
      1) detect near-zero columns from X_dirty;
      2) let k be the number detected and n be the original number of meters;
      3) use y_adjusted = y * (1 - k/n);
      4) exclude detected columns and fit the same estimators.

    Returns comparison metrics, per-user results, detection results, adjusted
    target, corruption metadata, and a one-row trial summary.
    """
    corruption_kwargs = dict(corruption_kwargs or {})
    dirty, alpha_true, corruption_metadata = make_corrupted_meter_data(
        X_clean, random_state=random_state, **corruption_kwargs
    )
    y = np.asarray(y, dtype=float).reshape(-1)
    X_arr = np.asarray(dirty, dtype=float)
    names = list(dirty.columns) if isinstance(dirty, pd.DataFrame) else [
        f"user_{j}" for j in range(X_arr.shape[1])]
    alpha_true = np.asarray(alpha_true, dtype=float)
    n_rows, n_total = X_arr.shape
    if len(y) != n_rows:
        raise ValueError("y must have the same number of rows as X_clean.")
    if methods is None:
        methods = default_comparison_methods()

    # Detection uses only the corrupted readings, not the known synthetic labels.
    detection_results = detect_unworking_meters(
        dirty, near_zero_fraction=near_zero_fraction, abs_tol=near_zero_abs_tol
    )
    detected_mask = detection_results["detected_unworking"].to_numpy(dtype=bool)
    detected_names = [names[j] for j in np.flatnonzero(detected_mask)]
    working_mask_true = alpha_true > 0
    n_detected = int(detected_mask.sum())
    y_adjusted = y * (1.0 - n_detected / n_total)
    keep_mask = ~detected_mask
    kept_names = [names[j] for j in np.flatnonzero(keep_mask)]

    true_unworking = np.asarray(corruption_metadata["unworking_mask"], dtype=bool)
    tp = int(np.sum(detected_mask & true_unworking))
    fp = int(np.sum(detected_mask & ~true_unworking))
    fn = int(np.sum(~detected_mask & true_unworking))
    detection_precision = tp / (tp + fp) if (tp + fp) else np.nan
    detection_recall = tp / (tp + fn) if (tp + fn) else np.nan
    detection_f1 = (2 * detection_precision * detection_recall /
                    (detection_precision + detection_recall)
                    if np.isfinite(detection_precision) and
                    np.isfinite(detection_recall) and
                    (detection_precision + detection_recall) > 0 else np.nan)
    detection_results["true_unworking"] = true_unworking
    detection_results["detection_correct"] = (
        detection_results["detected_unworking"].to_numpy() == true_unworking)
    detection_results.insert(0, "trial_id", trial_id)
    detection_results["random_state"] = random_state

    # Variant 1: original pipeline, all columns and unadjusted target.
    raw_metrics, raw_users = compare_alpha_estimators(
        dirty, y, alpha_true,
        methods=methods,
        bootstrap_reps=bootstrap_reps,
        block_length=block_length,
        random_state=random_state + 1,
        evaluation_mask=working_mask_true,
    )
    raw_metrics["pipeline_variant"] = "unadjusted_all_meters"
    raw_users["pipeline_variant"] = "unadjusted_all_meters"
    raw_metrics["target_adjustment"] = "none"
    raw_metrics["n_meters_used"] = n_total

    # Variant 2: equal-share correction and detected-column exclusion.
    if not keep_mask.any():
        raise ValueError("All meters were detected as unworking; no predictors remain.")
    X_kept = dirty.loc[:, keep_mask] if isinstance(dirty, pd.DataFrame) else X_arr[:, keep_mask]
    alpha_kept = alpha_true[keep_mask]
    kept_eval_mask = alpha_kept > 0
    adjusted_metrics, adjusted_users = compare_alpha_estimators(
        X_kept, y_adjusted, alpha_kept,
        methods=methods,
        bootstrap_reps=bootstrap_reps,
        block_length=block_length,
        random_state=random_state + 2,
        evaluation_mask=kept_eval_mask,
    )
    adjusted_metrics["pipeline_variant"] = "equal_share_adjusted"
    adjusted_users["pipeline_variant"] = "equal_share_adjusted"
    adjusted_metrics["target_adjustment"] = f"y*(1-{n_detected}/{n_total})"
    adjusted_metrics["n_meters_used"] = int(keep_mask.sum())

    for frame in (raw_metrics, adjusted_metrics):
        frame["trial_id"] = trial_id
        frame["random_state"] = random_state
        frame["detected_unworking_count"] = n_detected
        frame["detection_precision"] = detection_precision
        frame["detection_recall"] = detection_recall
        frame["detection_f1"] = detection_f1
    for frame in (raw_users, adjusted_users):
        frame["trial_id"] = trial_id
        frame["random_state"] = random_state
        frame["detected_unworking_count"] = n_detected

    method_metrics = pd.concat([raw_metrics, adjusted_metrics], ignore_index=True, sort=False)
    user_results = pd.concat([raw_users, adjusted_users], ignore_index=True, sort=False)
    trial_summary = pd.DataFrame([{
        "trial_id": trial_id,
        "random_state": random_state,
        "n_rows": n_rows,
        "n_users": n_total,
        "n_unworking_true": int(true_unworking.sum()),
        "n_unworking_detected": n_detected,
        "detection_precision": detection_precision,
        "detection_recall": detection_recall,
        "detection_f1": detection_f1,
        "n_scaled_users": len(corruption_metadata["scaled_columns"]),
        "n_block_corrupted_users": len(corruption_metadata["block_columns"]),
        "n_periodic_corrupted_users": len(corruption_metadata["periodic_columns"]),
        "equal_share_fraction_removed": n_detected / n_total,
    }])

    return {
        "X_dirty": dirty,
        "alpha_true": pd.Series(alpha_true, index=names, name="alpha_true"),
        "corruption_metadata": corruption_metadata,
        "detection_results": detection_results,
        "y_adjusted": pd.Series(y_adjusted, index=(dirty.index if isinstance(dirty, pd.DataFrame) else None), name="y_adjusted"),
        "detected_unworking_columns": detected_names,
        "kept_columns": kept_names,
        "method_metrics": method_metrics,
        "user_results": user_results,
        "trial_summary": trial_summary,
    }


def run_repeated_auto_tests(
    X_clean, y, *, n_trials=10, methods=None,
    corruption_kwargs=None, bootstrap_reps=100, block_length=24,
    base_seed=42, near_zero_fraction=0.98, near_zero_abs_tol=1e-8,
):
    """Run repeated trials and concatenate all result tables, including detection."""
    metric_tables, user_tables, detection_tables, trial_tables = [], [], [], []
    last_run = None
    for trial in range(n_trials):
        result = run_auto_test(
            X_clean, y,
            methods=methods,
            trial_id=trial,
            corruption_kwargs=corruption_kwargs,
            bootstrap_reps=bootstrap_reps,
            block_length=block_length,
            random_state=base_seed + trial,
            near_zero_fraction=near_zero_fraction,
            near_zero_abs_tol=near_zero_abs_tol,
        )
        metric_tables.append(result["method_metrics"])
        user_tables.append(result["user_results"])
        detection = result["detection_results"].copy()
        detection["trial_id"] = trial
        detection["random_state"] = base_seed + trial
        detection_tables.append(detection)
        trial_tables.append(result["trial_summary"])
        last_run = result

    return {
        "method_metrics": pd.concat(metric_tables, ignore_index=True),
        "user_results": pd.concat(user_tables, ignore_index=True),
        "detection_results": pd.concat(detection_tables, ignore_index=True),
        "trial_summary": pd.concat(trial_tables, ignore_index=True),
        "last_run": last_run,
    }


## Example: one trial, then repeated trials

`X_clean` must contain only meter columns (not `target`); `y_target` is the central-meter target generated from the clean readings plus noise. Set `unworking_share` to simulate meters whose entire columns are permanently zero. The pipeline reports both the original unadjusted fit and the equal-share baseline.

The equal-share baseline uses the **detected** number of unworking meters, not the true count. `near_zero_fraction` and `near_zero_abs_tol` control detection. Use a tolerance appropriate to the units and noise floor of the meter readings.

In [ ]:
# Ensure the clean feature matrix and target match the synthetic generation process.
X_clean = df_clean.drop(columns="target")
y_target = df_clean["target"].to_numpy()

methods_to_compare = default_comparison_methods(
    beta_min=1.0,
    beta_max=15.0,
    include_bayesian=True,  # set True to include slower PyMC sampling
)

# Quick smoke test. A small positive unworking_share creates permanently zeroed columns.
comparison = run_auto_test(
    X_clean,
    y_target,
    methods=methods_to_compare,
    trial_id=0,
    corruption_kwargs={
        "unworking_share": 0.05,
        "scale_share": 0.05,
        "scale_range": (0.2, 1.0),
        "block_share": 0.05,
        "block_fraction": 0.20,
        "periodic_share": 0.05,
        "every_k": 3,
    },
    near_zero_fraction=0.98,
    near_zero_abs_tol=1e-8,
    bootstrap_reps=20,
    block_length=24,
    random_state=42,
)

display(comparison["trial_summary"])
display(comparison["method_metrics"].sort_values(["pipeline_variant", "alpha_mae"]))
display(comparison["detection_results"])
display(comparison["user_results"].head())


 Progress                       Draw   Divergences   Step size   Grad evals   Speed          Elapsed   Remaining  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━   1000   0             0.194       15           9.54 draws/s   0:01:44   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━   1000   0             0.205       15           4.87 draws/s   0:03:25   0:00:00

,trial_id,random_state,n_rows,n_users,n_unworking_true,n_unworking_detected,detection_precision,detection_recall,detection_f1,n_scaled_users,n_block_corrupted_users,n_periodic_corrupted_users,equal_share_fraction_removed
0,0,42,8784,53,3,5,0.6,1.0,0.75,3,3,3,0.09434


,method,mse_in_sample,rmse_in_sample,r2_in_sample,alpha_mae,alpha_rmse,alpha_bias,alpha_interval_coverage,mean_alpha_interval_width,bootstrap_reps_requested,...,fit_message,pipeline_variant,target_adjustment,n_meters_used,trial_id,random_state,detected_unworking_count,detection_precision,detection_recall,detection_f1
16,L-BFGS-B_beta_bounds_no_intercept,52.255752,7.228814,0.970063,0.003972,0.024863,0.003972,0.958333,0.000000e+00,20,...,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
17,PyMC_Bayesian,52.826564,7.268188,0.969736,0.005562,0.025735,0.002116,0.000000,8.356872e-03,20,...,PyMC posterior mean beta,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
9,OLS_no_intercept,22.811907,4.776181,0.986931,0.417878,1.641907,0.285915,0.666667,1.198015e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
12,Ridge_0.1_no_intercept,22.811909,4.776181,0.986931,0.418368,1.643453,0.286504,0.562500,1.580839e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
13,Ridge_1_no_intercept,22.812096,4.776201,0.986931,0.422876,1.657137,0.291893,0.666667,2.366689e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
14,Ridge_10_no_intercept,22.821516,4.777187,0.986925,0.490576,1.816829,0.367433,0.625000,1.546091e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
11,OLS_positive_intercept,22.607793,4.754765,0.987048,1.742931,10.440591,1.653238,0.604167,5.878980e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
10,OLS_positive_no_intercept,22.909973,4.786436,0.986875,1.796484,10.856656,1.663801,0.562500,6.657639e+01,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
15,Ridge_positive_1_no_intercept,23.005182,4.796372,0.986820,2.170681,11.294155,2.067521,0.583333,3.697982e+01,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
8,PyMC_Bayesian,41.214731,6.419870,0.980633,0.071069,0.112276,-0.068295,0.020000,1.504613e-01,20,...,PyMC posterior mean beta,unadjusted_all_meters,none,53,0,42,5,0.6,1.0,0.75


,trial_id,user_id,near_zero_fraction,max_abs_reading,detected_unworking,true_unworking,detection_correct,random_state
0,0,12958209304783,0.008652,13.931133,False,False,True,42
1,0,51356699,0.000342,9.069219,False,False,True,42
2,0,12958209304417,0.000911,12.505150,False,False,True,42
3,0,12958209305151,0.014686,13.006777,False,False,True,42
4,0,12958209304688,1.000000,0.000000,True,True,True,42
5,0,51356695,0.001708,2.137960,False,False,True,42
6,0,13042207127896,0.101548,0.356066,False,False,True,42
7,0,13042207128171,0.200820,1.359950,False,False,True,42
8,0,51356602,0.000342,7.181747,False,False,True,42
9,0,13042207127923,0.001252,4.687286,False,False,True,42


,method,user_id,beta_hat,beta_boot_lower_95,beta_boot_upper_95,alpha_true,alpha_hat,alpha_boot_lower_95,alpha_boot_upper_95,alpha_evaluated,alpha_error,alpha_abs_error,alpha_interval_covers_truth,intercept_hat,fit_success,pipeline_variant,trial_id,random_state,detected_unworking_count
0,OLS_no_intercept,12958209304783,1.118090e+00,1.034356e+00,1.186100e+00,1.0,0.894383,8.433352e-01,9.668573e-01,True,-0.105617,0.105617,False,0.0,True,unadjusted_all_meters,0,42,5
1,OLS_no_intercept,51356699,1.121613e+00,1.021879e+00,1.194243e+00,1.0,0.891573,8.373684e-01,9.788305e-01,True,-0.108427,0.108427,False,0.0,True,unadjusted_all_meters,0,42,5
2,OLS_no_intercept,12958209304417,1.009889e+00,9.169838e-01,1.095840e+00,1.0,0.990208,9.127196e-01,1.090597e+00,True,-0.009792,0.009792,True,0.0,True,unadjusted_all_meters,0,42,5
3,OLS_no_intercept,12958209305151,1.120068e+00,9.807381e-01,1.204973e+00,1.0,0.892803,8.302945e-01,1.019641e+00,True,-0.107197,0.107197,True,0.0,True,unadjusted_all_meters,0,42,5
4,OLS_no_intercept,12958209304688,-1.965095e-13,-2.418760e-13,4.263562e-13,0.0,NaN,2.417503e+12,3.366327e+14,False,NaN,NaN,False,0.0,True,unadjusted_all_meters,0,42,5


## Important interpretation notes

1. **Weights and alpha are not the same thing.** The regressions and L-BFGS-B estimate `beta` in `y ≈ X_dirty @ beta + intercept`; `alpha_hat = 1 / beta_hat` is an implied scaling estimate. For methods with negative or zero weights, alpha is undefined and is kept as NaN.
2. **Bounds must match the parameterization.** If the intended alpha range is `[alpha_min, alpha_max]` with positive alpha, then the corresponding beta bounds are `[1/alpha_max, 1/alpha_min]`. For the assumption `alpha in (0, 1]`, beta is at least 1. A beta upper bound of 15 implies alpha is at least `1/15`, not that alpha can be zero.
3. **Ridge's `alpha` argument is a regularization strength**, not the meter scaling factor. Its coefficients are not constrained to the physically valid range unless constraints are explicitly imposed.
4. **Uncertainty estimates differ by method.** Bayesian credible intervals come from the posterior under its priors and likelihood. Bootstrap intervals for optimization/OLS/Ridge approximate sampling variability by refitting on resampled time blocks. They are not directly identical notions of uncertainty.
5. **Do not compare only R²/MSE.** The central total may be predicted well while individual user factors are wrong due to correlated columns. Primary metrics here are alpha MAE/RMSE/bias and interval coverage; predictive metrics are secondary.
6. **Temporary zeroing is not solved by this baseline.** The persistent `alpha_true` is preserved by the generator, but methods are still fitted on data containing zeroed intervals. This tests robustness to those outages; it does not explicitly model missing consumption. A follow-up comparison can fit only valid rows or add an explicit missingness model.
7. **Use the true generator multipliers, not `clean / dirty`, as ground truth.** For zeroed values, division is undefined. The generator knows the multiplier before applying temporary outages and saves it directly.


## Save result datasets for later analysis

The output includes method-level metrics, per-user alpha estimates, detection diagnostics for each meter, and trial summaries. For repeated trials, `detection_results.csv` records the true synthetic status for evaluation alongside the detection decision.

In [ ]:
# Save this run's result tables. Set append=True for accumulating trials.
def save_comparison_results(result, output_dir="model_comparison_results", append=True):
    from pathlib import Path
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    tables = {
        "method_metrics.csv": result["method_metrics"],
        "user_results.csv": result["user_results"],
        "trial_summary.csv": result["trial_summary"],
    }
    if "detection_results" in result:
        tables["detection_results.csv"] = result["detection_results"]
    for filename, table in tables.items():
        file_path = output_dir / filename
        write_header = not (append and file_path.exists())
        table.to_csv(file_path, mode="a" if append else "w",
                     header=write_header, index=False)
    return str(output_dir.resolve())

# Example:
save_comparison_results(comparison, append=True)


'/content/model_comparison_results'

## Interpretation notes for the equal-share baseline

- The rule `y_adjusted = y * (1 - k_detected / n_total)` assumes each meter contributes an equal share of the total on average. It is a baseline, not a physically guaranteed correction.
- A meter is detected as persistently unworking when at least `near_zero_fraction` of its readings have absolute value at most `near_zero_abs_tol`. This can mistake naturally inactive meters for failures, so inspect `detection_results`.
- Alpha-recovery metrics are computed only for meters with true `alpha_true > 0`; the zero alpha of an unworking meter cannot be recovered by taking `1 / beta`. The true status is used only to evaluate the synthetic experiment, not by the detection rule.
- Compare `pipeline_variant == "unadjusted_all_meters"` with `pipeline_variant == "equal_share_adjusted"`. Their prediction metrics use different targets (original versus adjusted), so prioritize alpha-recovery metrics for the main comparison.


In [ ]:
comparison["method_metrics"].sort_values(['alpha_mae'])

,method,mse_in_sample,rmse_in_sample,r2_in_sample,alpha_mae,alpha_rmse,alpha_bias,alpha_interval_coverage,mean_alpha_interval_width,bootstrap_reps_requested,...,fit_message,pipeline_variant,target_adjustment,n_meters_used,trial_id,random_state,detected_unworking_count,detection_precision,detection_recall,detection_f1
16,L-BFGS-B_beta_bounds_no_intercept,52.255752,7.228814,0.970063,0.003972,0.024863,0.003972,0.958333,0.000000e+00,20,...,CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
17,PyMC_Bayesian,52.826564,7.268188,0.969736,0.005562,0.025735,0.002116,0.000000,8.356872e-03,20,...,PyMC posterior mean beta,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
8,PyMC_Bayesian,41.214731,6.419870,0.980633,0.071069,0.112276,-0.068295,0.020000,1.504613e-01,20,...,PyMC posterior mean beta,unadjusted_all_meters,none,53,0,42,5,0.6,1.0,0.75
7,L-BFGS-B_beta_bounds_no_intercept,41.039653,6.406220,0.980715,0.082567,0.161105,-0.068078,0.740000,1.271686e-01,20,...,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,unadjusted_all_meters,none,53,0,42,5,0.6,1.0,0.75
0,OLS_no_intercept,27.811913,5.273700,0.986931,0.403743,1.473580,0.164994,0.540000,2.588659e+14,20,...,ok,unadjusted_all_meters,none,53,0,42,5,0.6,1.0,0.75
3,Ridge_0.1_no_intercept,27.811915,5.273700,0.986931,0.404176,1.474959,0.165527,0.560000,1.792242e+00,20,...,ok,unadjusted_all_meters,none,53,0,42,5,0.6,1.0,0.75
4,Ridge_1_no_intercept,27.812144,5.273722,0.986931,0.408161,1.487155,0.170407,0.520000,3.820836e+00,20,...,ok,unadjusted_all_meters,none,53,0,42,5,0.6,1.0,0.75
9,OLS_no_intercept,22.811907,4.776181,0.986931,0.417878,1.641907,0.285915,0.666667,1.198015e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
12,Ridge_0.1_no_intercept,22.811909,4.776181,0.986931,0.418368,1.643453,0.286504,0.562500,1.580839e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
13,Ridge_1_no_intercept,22.812096,4.776201,0.986931,0.422876,1.657137,0.291893,0.666667,2.366689e+00,20,...,ok,equal_share_adjusted,y*(1-5/53),48,0,42,5,0.6,1.0,0.75
